# SCENIC+ eGRN Analysis — NK Compartment

**Project**: P697 TEAseq T1D Low-Dose IL-2

**Date**: 2026-04-15

**Objective**: Infer enhancer-driven gene regulatory networks (eGRNs) for NK cells
using SCENIC+, then compare TF activity (eRegulon AUCell scores) across treatment
phases (Baseline → Post-IL2 → Post-RAPA → Followup) for longitudinal analysis.

**Input**: `data/outputData/scenic_export_NK/` (exported from `P697_DA.rmd`)
— RNA counts, ATAC counts, metadata, embeddings + MACS2 consensus peaks + fragment files.

**Environment**: `scenicplus_env` conda environment (see `code/scenicplus_environment.yml`)

**Pipeline**:
1. Load & prepare RNA AnnData (scanpy preprocessing)
2. Build pycisTopic object from ATAC data (topic modeling)
3. Generate region sets (binarized topics + DARs)
4. Run SCENIC+ SnakeMake pipeline (motif enrichment → GRN inference → AUCell)
5. Longitudinal TF activity comparison across treatment phases
6. Visualization & export

## 0. Setup

In [ ]:
import os
import time
import pickle
from datetime import datetime

import numpy as np
import pandas as pd
import scipy.io
import scipy.sparse
import scanpy as sc
import anndata as ad
import mudata as md
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc

import pycisTopic
import scenicplus

import session_info

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (8, 6)

print(f"scenicplus {scenicplus.__version__}")
print(f"pycisTopic {pycisTopic.__version__}")
print(f"scanpy {sc.__version__}")
print(f"anndata {ad.__version__}")

In [ ]:
# --- Paths ---
COMP = "NK"
PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
EXPORT_DIR = os.path.join(PROJECT_DIR, "data", "outputData", f"scenic_export_{COMP}")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "data", "outputData", f"scenicplus_{COMP}")
TABLE_DIR = os.path.join(PROJECT_DIR, "tables")
FIG_DIR = os.path.join(PROJECT_DIR, "figures", "scenicplus")
DB_DIR = os.path.join(PROJECT_DIR, "data", "inputData", "cisTarget_dbs")
MACS_DIR = os.path.join(PROJECT_DIR, "data", "outputData", "macs2_peaks")
FRAG_DIR = os.path.join(PROJECT_DIR, "data", "outputData")

for d in [OUTPUT_DIR, TABLE_DIR, FIG_DIR, DB_DIR]:
    os.makedirs(d, exist_ok=True)

TODAY = datetime.today().strftime("%Y-%m-%d")
print(f"Compartment: {COMP}")
print(f"Export dir:  {EXPORT_DIR}")
print(f"Output dir:  {OUTPUT_DIR}")
print(f"Date stamp:  {TODAY}")

## 0.1 cisTarget Database Selection

**Preferred**: a custom cisTarget DB built from this project's MACS2 consensus peaks
(`code/scenicplus_customdb/build_customdb_macs2.sh NK`). This is faster and more specific
than the 46 GB public hg38 screen v10 database. See that script's header for prereqs.

**Fallback**: download the prebuilt hg38 screen v10 clust database. Only needed if the
custom DB above has not been built. Skip if files already exist locally.


In [ ]:
# --- cisTarget database paths ---
# Default: use the custom DB built from MACS2 consensus peaks for this compartment.
# Build once with: bash code/scenicplus_customdb/build_customdb_macs2.sh NK
CUSTOM_DB_DIR = os.path.join(PROJECT_DIR, "data", "inputData", "cisTarget_dbs_custom")
CUSTOM_PREFIX = f"{COMP}_MACS2_peaks_v10nr_hg38"

custom_rankings = os.path.join(CUSTOM_DB_DIR, f"{CUSTOM_PREFIX}.regions_vs_motifs.rankings.feather")
custom_scores   = os.path.join(CUSTOM_DB_DIR, f"{CUSTOM_PREFIX}.regions_vs_motifs.scores.feather")
# Motif-to-TF annotation table is dataset-independent (reuse across NK/Treg).
motif_annot_fname = os.path.join(
    PROJECT_DIR, "data", "inputData", "aertslab_motif_collection",
    "v10nr_clust_public", "snapshots",
    "motifs-v10-nr.hgnc-m0.00001-o0.0.tbl",
)

USE_CUSTOM_DB = os.path.exists(custom_rankings) and os.path.exists(custom_scores)

if USE_CUSTOM_DB:
    rankings_fname = custom_rankings
    scores_fname   = custom_scores
    print("Using CUSTOM cisTarget DB built from MACS2 consensus peaks:")
    print(f"  rankings: {rankings_fname}")
    print(f"  scores:   {scores_fname}")
else:
    # --- Fallback: download prebuilt public hg38 screen v10 clust database (~46 GB) ---
    print("Custom DB not found; falling back to prebuilt public hg38 screen v10 clust DB.")
    RANKINGS_URL = "https://resources.aertslab.org/cistarget/databases/homo_sapiens/hg38/screen/mc_v10_clust/region_based/hg38_screen_v10_clust.regions_vs_motifs.rankings.feather"
    SCORES_URL   = "https://resources.aertslab.org/cistarget/databases/homo_sapiens/hg38/screen/mc_v10_clust/region_based/hg38_screen_v10_clust.regions_vs_motifs.scores.feather"
    MOTIF_ANNOT_URL = "https://resources.aertslab.org/cistarget/motif2tf/motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl"

    rankings_fname = os.path.join(DB_DIR, "hg38_screen_v10_clust.regions_vs_motifs.rankings.feather")
    scores_fname   = os.path.join(DB_DIR, "hg38_screen_v10_clust.regions_vs_motifs.scores.feather")
    motif_annot_fname = os.path.join(DB_DIR, "motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl")

    for url, fname in [
        (RANKINGS_URL, rankings_fname),
        (SCORES_URL, scores_fname),
        (MOTIF_ANNOT_URL, motif_annot_fname),
    ]:
        if not os.path.exists(fname):
            print(f"Downloading {os.path.basename(fname)}...")
            !wget -q -O "{fname}" "{url}"
            print(f"  Saved: {fname} ({os.path.getsize(fname) / 1e9:.1f} GB)")
        else:
            print(f"Already exists: {os.path.basename(fname)} ({os.path.getsize(fname) / 1e9:.1f} GB)")

assert os.path.exists(motif_annot_fname), (
    f"Motif annotation table missing: {motif_annot_fname}. "
    "Download from https://resources.aertslab.org/cistarget/motif2tf/ if needed."
)


---
## 1. Load Data & Construct RNA AnnData

In [ ]:
# --- RNA counts (sparse) ---
rna_counts = scipy.io.mmread(os.path.join(EXPORT_DIR, "RNA_counts.mtx")).T.tocsr()
rna_genes = pd.read_csv(os.path.join(EXPORT_DIR, "RNA_genes.txt"), header=None)[0].values
rna_barcodes = pd.read_csv(os.path.join(EXPORT_DIR, "RNA_barcodes.txt"), header=None)[0].values

print(f"RNA matrix: {rna_counts.shape[0]} cells x {rna_counts.shape[1]} genes")
assert rna_counts.shape == (len(rna_barcodes), len(rna_genes))

In [ ]:
# --- Metadata ---
meta = pd.read_csv(os.path.join(EXPORT_DIR, "metadata.csv"), index_col=0)
meta.index.name = None

# Rename Washout → Followup if needed
if "Washout" in meta["treatmentPhase"].values:
    meta["treatmentPhase"] = meta["treatmentPhase"].replace("Washout", "Followup")
    print("Renamed Washout → Followup")

meta = meta.loc[rna_barcodes]

print("\nCells per donor × treatmentPhase:")
ct = pd.crosstab(meta["donorID"], meta["treatmentPhase"])
print(ct)
print(f"\nTotal cells: {len(meta)}")

In [ ]:
# --- Construct RNA AnnData ---
adata_rna = ad.AnnData(
    X=rna_counts,
    obs=meta.copy(),
    var=pd.DataFrame(index=rna_genes),
)
adata_rna.obs_names = rna_barcodes.tolist()
adata_rna.var_names_make_unique()

# Categorical types
adata_rna.obs["treatmentPhase"] = pd.Categorical(
    adata_rna.obs["treatmentPhase"],
    categories=["Baseline", "Post-IL2", "Post-RAPA", "Followup"],
    ordered=True,
)
adata_rna.obs["wnn_clusters"] = adata_rna.obs["wnn_clusters"].astype(str).astype("category")
adata_rna.obs["donorID"] = adata_rna.obs["donorID"].astype(str).astype("category")

print(adata_rna)

In [ ]:
# --- Scanpy preprocessing (required by SCENIC+) ---
# Save raw counts
adata_rna.raw = adata_rna

# Normalize + log-transform
sc.pp.normalize_total(adata_rna, target_sum=1e4)
sc.pp.log1p(adata_rna)

# HVG selection
sc.pp.highly_variable_genes(adata_rna, min_mean=0.0125, max_mean=3, min_disp=0.5)
print(f"Highly variable genes: {adata_rna.var['highly_variable'].sum()}")

# Subset to HVGs for PCA/neighbor calculation
adata_rna_hvg = adata_rna[:, adata_rna.var.highly_variable].copy()
sc.pp.scale(adata_rna_hvg, max_value=10)
sc.tl.pca(adata_rna_hvg)
sc.pp.neighbors(adata_rna_hvg)
sc.tl.umap(adata_rna_hvg)

# Transfer UMAP coordinates back to full AnnData
adata_rna.obsm["X_umap"] = adata_rna_hvg.obsm["X_umap"]

# Save preprocessed RNA AnnData
rna_h5ad_path = os.path.join(OUTPUT_DIR, f"{COMP}_adata_rna.h5ad")
adata_rna.write(rna_h5ad_path)
print(f"Saved RNA AnnData: {rna_h5ad_path}")

In [ ]:
# Quick sanity check: UMAP colored by treatment phase
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc.pl.umap(adata_rna, color="treatmentPhase", ax=axes[0], show=False, title="Treatment Phase")
sc.pl.umap(adata_rna, color="wnn_clusters", ax=axes[1], show=False, title="WNN Clusters")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"P697.{TODAY}_{COMP}_scenicplus_rna_umap.pdf"), bbox_inches="tight")
plt.show()

---
## 2. Build pycisTopic Object from ATAC Data

In [ ]:
# --- ATAC counts (sparse) ---
atac_counts = scipy.io.mmread(os.path.join(EXPORT_DIR, "ATAC_counts.mtx")).T.tocsr()
atac_peaks = pd.read_csv(os.path.join(EXPORT_DIR, "ATAC_peaks.txt"), header=None)[0].values
atac_barcodes = pd.read_csv(os.path.join(EXPORT_DIR, "ATAC_barcodes.txt"), header=None)[0].values

print(f"ATAC matrix: {atac_counts.shape[0]} cells x {atac_counts.shape[1]} peaks")
assert np.array_equal(rna_barcodes, atac_barcodes), "RNA and ATAC barcodes must match"

In [ ]:
# --- TEST MODE: pseudo-random subsample for rapid iteration ---
# For a PRODUCTION run, either set TEST_MODE = False or skip this cell entirely.
# The subsample is stratified by treatmentPhase (fallback: uniform) to preserve
# class balance, and both RNA and ATAC matrices / metadata are pruned together
# so downstream cistopic, snakemake, and analysis cells operate on the subset.
TEST_MODE = True        # <-- flip to False for production
TEST_FRAC = 0.05        # fraction of cells to retain
TEST_SEED = 42

if TEST_MODE:
    rng = np.random.default_rng(TEST_SEED)
    n_before = len(rna_barcodes)

    # Stratified sample by treatmentPhase (keeps class balance at small N)
    phase_series = pd.Series(meta["treatmentPhase"].astype(str).values, index=np.arange(n_before))
    keep_idx = []
    for ph, idxs in phase_series.groupby(phase_series).groups.items():
        idxs = np.asarray(idxs)
        n_keep = max(1, int(round(len(idxs) * TEST_FRAC)))
        keep_idx.append(rng.choice(idxs, size=n_keep, replace=False))
    keep_idx = np.sort(np.concatenate(keep_idx))

    # Apply to all aligned objects
    rna_barcodes  = rna_barcodes[keep_idx]
    atac_barcodes = atac_barcodes[keep_idx]
    meta          = meta.iloc[keep_idx].copy()
    adata_rna     = adata_rna[keep_idx, :].copy()
    atac_counts   = atac_counts[keep_idx, :]

    # Re-assert barcode alignment after subsampling
    assert np.array_equal(rna_barcodes, atac_barcodes), "Subsample broke barcode alignment"
    assert adata_rna.n_obs == len(rna_barcodes) == atac_counts.shape[0]

    # Also persist the pruned RNA AnnData so the snakemake pipeline consumes the subset
    rna_h5ad_path = os.path.join(OUTPUT_DIR, f"{COMP}_adata_rna.h5ad")
    adata_rna.write(rna_h5ad_path)

    print(f"[TEST_MODE] Subsampled: {n_before} -> {len(rna_barcodes)} cells "
          f"({TEST_FRAC*100:.1f}%, seed={TEST_SEED}).")

    # Warn if any cluster x phase bin is thinly populated (metacells + stats may be unreliable)
    bin_counts = pd.crosstab(meta["wnn_clusters"], meta["treatmentPhase"])
    thin = bin_counts[bin_counts < 20]
    thin = thin.stack().dropna()
    if len(thin) > 0:
        print(f"[TEST_MODE] WARNING: {len(thin)} wnn_clusters x treatmentPhase bins have <20 cells; "
              "metacell aggregation and Kruskal-Wallis stats may be underpowered.")
        print(bin_counts)
else:
    print("TEST_MODE disabled; using full dataset.")


In [ ]:
# --- Create cisTopic object from existing count matrix ---
from pycisTopic.cistopic_class import CistopicObject

# pycisTopic expects: regions x cells (transposed from our cells x regions)
cistopic_obj = CistopicObject(
    fragment_matrix=atac_counts.T,  # regions x cells
    cell_names=atac_barcodes.tolist(),
    region_names=atac_peaks.tolist(),
    project=f"P697_{COMP}",
)

# Add cell metadata
meta_for_cistopic = meta.copy()
meta_for_cistopic.index = atac_barcodes.tolist()
cistopic_obj.add_cell_data(meta_for_cistopic)

print(cistopic_obj)

In [ ]:
# --- Cross-modal clustering QC (adapted from colleague's ArchR pipeline) ---
# Cluster the ATAC (topic) space at multiple Leiden resolutions and compare
# against the RNA-derived WNN clusters with Jaccard / ARI / NMI / purity.
# This sanity check flags cases where the ATAC modality partitions the cells
# very differently from the RNA / WNN reference before we run LDA + SCENIC+.
from pycisTopic.clust_vis import find_clusters, run_umap
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    homogeneity_score,
    completeness_score,
)

qc_resolutions = [0.6, 0.8, 2.0, 2.3, 2.6, 3.0, 4.0, 5.0, 6.0]
find_clusters(
    cistopic_obj,
    target="cell",
    k=15,
    res=qc_resolutions,
    prefix="pycisTopic_leiden_15_",
    scale=True,
    split_pattern="-",
)
run_umap(cistopic_obj, target="cell", scale=True)

ref_labels = meta.loc[cistopic_obj.cell_names, "wnn_clusters"].astype(str).values
qc_rows = []
for res in qc_resolutions:
    col = f"pycisTopic_leiden_15_{res}"
    if col not in cistopic_obj.cell_data.columns:
        continue
    atac_labels = cistopic_obj.cell_data[col].astype(str).values
    qc_rows.append({
        "resolution": res,
        "n_atac_clusters": len(np.unique(atac_labels)),
        "ARI": adjusted_rand_score(ref_labels, atac_labels),
        "NMI": normalized_mutual_info_score(ref_labels, atac_labels),
        "homogeneity": homogeneity_score(ref_labels, atac_labels),
        "completeness": completeness_score(ref_labels, atac_labels),
    })
qc_df = pd.DataFrame(qc_rows)
print(qc_df.round(3))

# Persist QC table + quick heatmap of metrics across resolutions.
qc_csv = os.path.join(TABLE_DIR, f"P697.{TODAY}_{COMP}_scenicplus_clustering_QC.csv")
qc_df.to_csv(qc_csv, index=False)

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(
    qc_df.set_index("resolution")[["ARI", "NMI", "homogeneity", "completeness"]].T,
    cmap="viridis", annot=True, fmt=".2f", ax=ax,
)
ax.set_title(f"{COMP} — pycisTopic (Leiden k=15) vs WNN clusters")
plt.tight_layout()
plt.savefig(
    os.path.join(FIG_DIR, f"P697.{TODAY}_{COMP}_scenicplus_clustering_QC.pdf"),
    bbox_inches="tight",
)
plt.show()


### 2.1 LDA Topic Modeling

Run several topic models to find optimal number of topics.
Using the Mallet-based CGS implementation for better performance.

In [ ]:
# --- Download Mallet if needed ---
MALLET_DIR = os.path.join(PROJECT_DIR, "data", "inputData", "Mallet-202108")
mallet_path = os.path.join(MALLET_DIR, "bin", "mallet")

if not os.path.exists(mallet_path):
    print("Downloading Mallet...")
    !wget -q -O /tmp/Mallet-202108-bin.tar.gz https://github.com/mimno/Mallet/releases/download/v202108/Mallet-202108-bin.tar.gz
    !tar -xf /tmp/Mallet-202108-bin.tar.gz -C {os.path.join(PROJECT_DIR, 'data', 'inputData')}
    print(f"Mallet installed at: {mallet_path}")
else:
    print(f"Mallet already available: {mallet_path}")

In [ ]:
# --- Serialize cistopic_obj and hand LDA off to a terminal run ---
# Rationale: MALLET LDA is memory-hungry and routinely crashes Jupyter kernels
# mid-convergence. We run it from a standalone shell script with nohup (colleague's
# pattern from scenicplus_from_ArchR_and_Seuratobj). Expanded topic sweep
# [2, 10, 20, 30, 40, 50, 60] matches the colleague and gives evaluate_models a
# wider range than the previous notebook-only run.
preLDA_pkl_path = os.path.join(OUTPUT_DIR, f"{COMP}_cistopic_obj_preLDA.pkl")
models_pkl_path = os.path.join(OUTPUT_DIR, "models_all_topics.pkl")

with open(preLDA_pkl_path, "wb") as f:
    pickle.dump(cistopic_obj, f)
print(f"Saved pre-LDA cistopic object: {preLDA_pkl_path}")

print("\nNext step: run LDA OUTSIDE this notebook (prevents kernel crashes).")
print("In a terminal with the scenicplus conda env active:")
print("    conda activate scenicplus_env")
print(f"    nohup bash code/scenicplus_lda/run_lda_{COMP}.sh \\")
print(f"        > scenicplus_{COMP}_lda.log 2>&1 &")
print(f"    tail -f scenicplus_{COMP}_lda.log")
print(f"\nWhen the script writes {os.path.basename(models_pkl_path)}, continue "
      "with the next cell.")


In [ ]:
# --- Load terminal-produced LDA models & evaluate ---
from pycisTopic.lda_models import evaluate_models

assert os.path.exists(models_pkl_path), (
    f"LDA models pickle not found: {models_pkl_path}. "
    f"Run code/scenicplus_lda/run_lda_{COMP}.sh from a terminal first."
)
with open(models_pkl_path, "rb") as f:
    models = pickle.load(f)
print(f"Loaded {len(models)} topic models: "
      f"{sorted({m.n_topic for m in models})}")

model_metrics = evaluate_models(
    models,
    select_model=None,  # returns metrics for all
    return_model=False,
    metrics=["Arun_2010", "Cao_Juan_2009", "Minmo_2011", "loglikelihood"],
    plot_metrics=True,
)

plt.savefig(os.path.join(FIG_DIR, f"P697.{TODAY}_{COMP}_scenicplus_topic_model_metrics.pdf"), bbox_inches="tight")
plt.show()


In [ ]:
# --- Select best model ---
# TODO: TUNABLE — select n_topics based on metrics above
SELECTED_N_TOPICS = 15  # Adjust based on metrics plot

cistopic_obj = evaluate_models(
    models,
    select_model=SELECTED_N_TOPICS,
    return_model=True,
)

print(f"Selected model: {SELECTED_N_TOPICS} topics")
print(cistopic_obj)

### 2.2 Topic Binarization & Region Sets

In [ ]:
# --- Binarize topics ---
from pycisTopic.topic_binarization import binarize_topics

# Otsu method for topic-region binarization
region_bin_topics_otsu = binarize_topics(cistopic_obj, method="otsu")
# Top 3k regions per topic (balanced sets for motif enrichment)
region_bin_topics_top3k = binarize_topics(cistopic_obj, method="ntop", ntop=3000)

print("Otsu binarized topics:")
for topic, regions in region_bin_topics_otsu.items():
    print(f"  {topic}: {len(regions)} regions")

print(f"\nTop 3k binarized topics: {len(region_bin_topics_top3k)} topics")

In [ ]:
# --- Save region sets as BED files (required by SCENIC+ SnakeMake) ---
from pycisTopic.utils import region_names_to_coordinates

region_set_dir = os.path.join(OUTPUT_DIR, "region_sets")
topics_otsu_dir = os.path.join(region_set_dir, "topics_otsu")
topics_top3k_dir = os.path.join(region_set_dir, "topics_top_3k")
dars_dir = os.path.join(region_set_dir, "DARs_treatmentPhase")

for d in [topics_otsu_dir, topics_top3k_dir, dars_dir]:
    os.makedirs(d, exist_ok=True)

# Save Otsu binarized topic regions
for topic in region_bin_topics_otsu:
    region_names_to_coordinates(
        region_bin_topics_otsu[topic].index
    ).sort_values(
        ["Chromosome", "Start", "End"]
    ).to_csv(
        os.path.join(topics_otsu_dir, f"{topic}.bed"),
        sep="\t", header=False, index=False,
    )

# Save top-3k topic regions
for topic in region_bin_topics_top3k:
    region_names_to_coordinates(
        region_bin_topics_top3k[topic].index
    ).sort_values(
        ["Chromosome", "Start", "End"]
    ).to_csv(
        os.path.join(topics_top3k_dir, f"{topic}.bed"),
        sep="\t", header=False, index=False,
    )

print(f"Saved {len(region_bin_topics_otsu)} Otsu topic BEDs to {topics_otsu_dir}")
print(f"Saved {len(region_bin_topics_top3k)} Top-3k topic BEDs to {topics_top3k_dir}")

In [ ]:
# --- Add DARs from existing DA analysis as additional region sets ---
# Read per-cluster ATAC DA tables and extract significant DARs
import glob

dar_files = sorted(glob.glob(os.path.join(TABLE_DIR, f"P697.*_{COMP}_ATAC_DA_*.csv")))
# Also include per-cluster DARs
dar_files += sorted(glob.glob(os.path.join(TABLE_DIR, f"P697.*_{COMP}_cluster*_ATAC_DA_*.csv")))

print(f"Found {len(dar_files)} DAR files for {COMP}")

for dar_file in dar_files:
    basename = os.path.basename(dar_file).replace(".csv", "")
    # Extract contrast name (e.g., PostIL2_vs_BL)
    parts = basename.split("_ATAC_DA_")
    if len(parts) == 2:
        contrast_name = parts[1]
        prefix = parts[0].split(".")[-1]  # e.g., NK or NK_cluster0
        bed_name = f"DARs_{prefix}_{contrast_name}"
    else:
        bed_name = basename

    df = pd.read_csv(dar_file)
    # Filter significant DARs (FDR < 0.05, |logFC| > 0.5)
    if "adj.P.Val" in df.columns and "logFC" in df.columns and "feature" in df.columns:
        sig = df[(df["adj.P.Val"] < 0.05) & (df["logFC"].abs() > 0.5)]
        if len(sig) > 0:
            # Parse peak names to BED format (assumed chr-start-end)
            peak_coords = sig["feature"].str.split("-", expand=True)
            if peak_coords.shape[1] >= 3:
                bed_df = pd.DataFrame({
                    "Chromosome": peak_coords[0],
                    "Start": peak_coords[1].astype(int),
                    "End": peak_coords[2].astype(int),
                })
                bed_df.sort_values(["Chromosome", "Start", "End"]).to_csv(
                    os.path.join(dars_dir, f"{bed_name}.bed"),
                    sep="\t", header=False, index=False,
                )
                print(f"  {bed_name}: {len(sig)} sig DARs → BED")

print(f"\nTotal region set dirs: {len(os.listdir(region_set_dir))}")

In [ ]:
# --- Save cisTopic object ---
cistopic_pkl_path = os.path.join(OUTPUT_DIR, f"{COMP}_cistopic_obj.pkl")
pickle.dump(cistopic_obj, open(cistopic_pkl_path, "wb"))
print(f"Saved cisTopic object: {cistopic_pkl_path}")

---
## 3. Configure & Run SCENIC+ SnakeMake Pipeline

In [ ]:
# --- Initialize SCENIC+ SnakeMake pipeline ---
from scenicplus.cli.commands import init_snakemake

scplus_pipeline_dir = os.path.join(OUTPUT_DIR, "scplus_pipeline")

init_snakemake(
    out_dir=scplus_pipeline_dir,
)

In [ ]:
# --- Update SCENIC+ config ---
# Parameter values below mirror the colleague's production ArchR pipeline
# (scenicplus_from_ArchR_and_Seuratobj/snakemake_config/config_mergedcd8_ArchRpeaks.yaml)
# and the SCENIC+ tutorials. Explicit thresholds for DEM / CTX / inference
# replace library defaults so the run is fully reproducible from this config.
import yaml

config_path = os.path.join(scplus_pipeline_dir, "Snakemake", "config", "config.yaml")

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# --- Input data ---
config["input_data"]["cisTopic_obj_fname"] = cistopic_pkl_path
config["input_data"]["GEX_anndata_fname"] = rna_h5ad_path
config["input_data"]["region_set_folder"] = region_set_dir
config["input_data"]["ctx_db_fname"] = rankings_fname
config["input_data"]["dem_db_fname"] = scores_fname
config["input_data"]["path_to_motif_annotations"] = motif_annot_fname

# --- General params ---
config["params_general"]["temp_dir"] = os.path.join(OUTPUT_DIR, "tmp")
config["params_general"]["n_cpu"] = 20  # TODO: adjust for your Coder resources
config["params_general"]["seed"] = 42

# --- Data preparation ---
config["params_data_preparation"]["is_multiome"] = True
config["params_data_preparation"]["bc_transform_func"] = "\"lambda x: x\""  # barcodes already match
config["params_data_preparation"]["key_to_group_by"] = "wnn_clusters"
# TODO: consider lowering to 5 for rare clusters if any cluster x phase bin
# is sparse (see clustering-QC cell above). For now use the colleague's value.
config["params_data_preparation"]["nr_cells_per_metacells"] = 10

# --- Genome annotations ---
config["params_data_preparation"]["species"] = "hsapiens"
config["params_data_preparation"]["biomart_host"] = "http://www.ensembl.org"

# --- Search space ---
config["params_data_preparation"]["search_space_upstream"] = "1000 150000"
config["params_data_preparation"]["search_space_downstream"] = "1000 150000"
config["params_data_preparation"]["search_space_extend_tss"] = "10 10"

# --- Motif enrichment: DEM (differential) ---
config["params_motif_enrichment"]["species"] = "homo_sapiens"
config["params_motif_enrichment"]["annotation_version"] = "v10nr_clust"
config["params_motif_enrichment"]["motif_similarity_fdr"] = 0.001
config["params_motif_enrichment"]["orthologous_identity_threshold"] = 0.0
config["params_motif_enrichment"]["annotations_to_use"] = "Direct_annot Orthology_annot"
config["params_motif_enrichment"]["dem_adj_pval_thr"] = 0.05
config["params_motif_enrichment"]["dem_log2fc_thr"] = 1.0
config["params_motif_enrichment"]["dem_mean_fg_thr"] = 0.0
config["params_motif_enrichment"]["dem_motif_hit_thr"] = 3.0
config["params_motif_enrichment"]["dem_max_bg_regions"] = 500
config["params_motif_enrichment"]["dem_balance_number_of_promoters"] = True
config["params_motif_enrichment"]["dem_promoter_space"] = 1000
config["params_motif_enrichment"]["fraction_overlap_w_dem_database"] = 0.4

# --- Motif enrichment: CTX (cisTarget) ---
config["params_motif_enrichment"]["ctx_auc_threshold"] = 0.005
config["params_motif_enrichment"]["ctx_nes_threshold"] = 3.0
config["params_motif_enrichment"]["ctx_rank_threshold"] = 0.05
config["params_motif_enrichment"]["fraction_overlap_w_ctx_database"] = 0.4

# --- Inference (TF->gene, region->gene, eRegulon construction) ---
config["params_inference"]["tf_to_gene_importance_method"] = "GBM"
config["params_inference"]["region_to_gene_importance_method"] = "GBM"
config["params_inference"]["region_to_gene_correlation_method"] = "SR"
config["params_inference"]["gsea_n_perm"] = 1000
# Multi-level regulon construction (strict -> permissive). Produces 3 levels of
# confidence tied to top-N regions per gene for downstream filtering.
config["params_inference"]["quantile_thresholds_region_to_gene"] = "0.85 0.90 0.95"
config["params_inference"]["top_n_regionTogenes_per_gene"] = "5 10 15"
config["params_inference"]["min_target_genes"] = 10
config["params_inference"]["rho_threshold"] = 0.05

# Write updated config
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"Config written to: {config_path}")
print(f"\nKey settings:")
print(f"  n_cpu: {config['params_general']['n_cpu']}")
print(f"  cisTopic object: {os.path.basename(cistopic_pkl_path)}")
print(f"  RNA AnnData: {os.path.basename(rna_h5ad_path)}")
print(f"  Region sets: {region_set_dir}")
print(f"  cisTarget rankings: {os.path.basename(rankings_fname)}  (custom={USE_CUSTOM_DB})")
print(f"  Motif annotations: {os.path.basename(motif_annot_fname)}")


In [ ]:
# --- Snakemake dry-run: validate config + DAG before the real execution ---
snakemake_dir = os.path.join(scplus_pipeline_dir, "Snakemake")
print(f"Dry-run in: {snakemake_dir}\n")
!cd {snakemake_dir} && snakemake --cores 1 --dry-run 2>&1 | tail -n 80


In [ ]:
# --- Run SCENIC+ SnakeMake pipeline ---
# This runs all 13 jobs: motif enrichment → cistromes → R2G → TF2G → eGRN → AUCell
#
# Execution backend:
#   * BRI Coder workspaces with `enable_snakemake = true` get a pre-written
#     Kubernetes profile at ~/.config/snakemake/coder/config.yaml. We detect
#     it and farm rules out as separate K8s pods (much faster, lighter on
#     the workspace pod).
#   * Otherwise we fall back to local execution with --cores n_cpu.
snakemake_dir = os.path.join(scplus_pipeline_dir, "Snakemake")
coder_profile = os.path.expanduser("~/.config/snakemake/coder/config.yaml")
use_coder_profile = os.path.exists(coder_profile)

t0 = time.time()
print(f"Starting SCENIC+ pipeline at {datetime.now().strftime('%H:%M:%S')}...")
print(f"Working directory: {snakemake_dir}")
if use_coder_profile:
    print("Detected Coder Snakemake profile -> using Kubernetes executor")
    smk_cmd = (
        "snakemake --profile coder --jobs 20 --use-conda "
        "--latency-wait 120 --keep-going"
    )
else:
    print("Coder Snakemake profile not found -> running locally with --cores")
    smk_cmd = f"snakemake --cores {config['params_general']['n_cpu']} --use-conda"
print(f"Command: {smk_cmd}\n")

!cd {snakemake_dir} && {smk_cmd}

elapsed = time.time() - t0
print(f"\nSCENIC+ pipeline completed in {elapsed/60:.1f} min")

In [ ]:
# --- Verify pipeline outputs ---
output_dir_sm = os.path.join(scplus_pipeline_dir, "Snakemake")

expected_outputs = [
    config["output_data"]["scplus_mdata"],
    config["output_data"]["eRegulons_direct"],
    config["output_data"]["eRegulons_extended"],
    config["output_data"]["AUCell_direct"],
    config["output_data"]["AUCell_extended"],
    config["output_data"]["tf_to_gene_adjacencies"],
    config["output_data"]["region_to_gene_adjacencies"],
]

print("Pipeline output files:")
for fname in expected_outputs:
    fpath = os.path.join(scplus_pipeline_dir, fname)
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"  ✓ {fname} ({size_mb:.1f} MB)")
    else:
        print(f"  ✗ {fname} — MISSING")

---
## 4. Load SCENIC+ Results

In [ ]:
# --- Load final MuData with all eGRN results ---
scplus_mdata_path = os.path.join(scplus_pipeline_dir, config["output_data"]["scplus_mdata"])
scplus_mdata = md.read(scplus_mdata_path)

print(scplus_mdata)
print(f"\nModalities: {list(scplus_mdata.mod.keys())}")
for mod_name, mod_data in scplus_mdata.mod.items():
    print(f"  {mod_name}: {mod_data.shape}")

In [ ]:
# --- Extract eRegulon metadata ---
eRegulon_direct = pd.read_csv(
    os.path.join(scplus_pipeline_dir, config["output_data"]["eRegulons_direct"]),
    sep="\t",
)
eRegulon_extended = pd.read_csv(
    os.path.join(scplus_pipeline_dir, config["output_data"]["eRegulons_extended"]),
    sep="\t",
)

print(f"Direct eRegulons: {eRegulon_direct.shape[0]} TF-region-gene triplets")
print(f"  Unique TFs: {eRegulon_direct['TF'].nunique()}")
print(f"  Unique target genes: {eRegulon_direct['Gene'].nunique()}")

print(f"\nExtended eRegulons: {eRegulon_extended.shape[0]} TF-region-gene triplets")
print(f"  Unique TFs: {eRegulon_extended['TF'].nunique()}")

In [ ]:
# --- Check for expected NK biology ---
# TFs expected in NK cells
nk_tfs = ["TBX21", "EOMES", "STAT5A", "STAT5B", "STAT4", "RUNX3", "ETS1", "NFKB1", "IRF1"]

found_direct = eRegulon_direct[eRegulon_direct["TF"].isin(nk_tfs)]["TF"].unique()
found_extended = eRegulon_extended[eRegulon_extended["TF"].isin(nk_tfs)]["TF"].unique()

print("NK-relevant TFs in direct eRegulons:", sorted(found_direct))
print("NK-relevant TFs in extended eRegulons:", sorted(found_extended))

# IL2-STAT5 pathway check
stat5_regulon = eRegulon_direct[eRegulon_direct["TF"].isin(["STAT5A", "STAT5B"])]
if len(stat5_regulon) > 0:
    print(f"\nSTAT5 eRegulon: {stat5_regulon['Gene'].nunique()} target genes")
    print(f"  Top target genes: {stat5_regulon.groupby('Gene').size().sort_values(ascending=False).head(10).index.tolist()}")
else:
    print("\n⚠ No STAT5 eRegulon found in direct annotations")

---
## 5. Longitudinal TF Activity Analysis

Compare eRegulon activity (AUCell scores) across treatment phases to identify
dynamically regulated TF programs.

In [ ]:
# --- Extract AUCell scores ---
# Gene-based AUCell scores (direct annotation)
aucell_gene_direct = scplus_mdata.mod["direct_gene_based_AUC"].copy()

# Add treatment phase metadata
cell_meta = meta.loc[aucell_gene_direct.obs_names].copy()
aucell_gene_direct.obs["treatmentPhase"] = pd.Categorical(
    cell_meta["treatmentPhase"].values,
    categories=["Baseline", "Post-IL2", "Post-RAPA", "Followup"],
    ordered=True,
)
aucell_gene_direct.obs["wnn_clusters"] = cell_meta["wnn_clusters"].values
aucell_gene_direct.obs["donorID"] = cell_meta["donorID"].values

print(f"AUCell matrix: {aucell_gene_direct.shape[0]} cells × {aucell_gene_direct.shape[1]} eRegulons")
print(f"\neRegulon names (first 20):")
print(aucell_gene_direct.var_names[:20].tolist())

In [ ]:
# --- UMAP on eRegulon activity scores ---
sc.pp.neighbors(aucell_gene_direct, use_rep="X")
sc.tl.umap(aucell_gene_direct)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

sc.pl.umap(aucell_gene_direct, color="treatmentPhase", ax=axes[0], show=False,
           title="eRegulon UMAP — Treatment Phase", size=15)
sc.pl.umap(aucell_gene_direct, color="wnn_clusters", ax=axes[1], show=False,
           title="eRegulon UMAP — WNN Clusters", size=15)
sc.pl.umap(aucell_gene_direct, color="donorID", ax=axes[2], show=False,
           title="eRegulon UMAP — Donor", size=15)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"P697.{TODAY}_{COMP}_eRegulon_UMAP.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# --- Differential eRegulon activity across treatment phases ---
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests

phase_order = ["Baseline", "Post-IL2", "Post-RAPA", "Followup"]
aucell_df = pd.DataFrame(
    aucell_gene_direct.X if isinstance(aucell_gene_direct.X, np.ndarray)
    else aucell_gene_direct.X.toarray(),
    index=aucell_gene_direct.obs_names,
    columns=aucell_gene_direct.var_names,
)
aucell_df["treatmentPhase"] = aucell_gene_direct.obs["treatmentPhase"].values

# Kruskal-Wallis test for each eRegulon
results = []
for ereg in aucell_gene_direct.var_names:
    groups = [aucell_df.loc[aucell_df["treatmentPhase"] == phase, ereg].values
              for phase in phase_order if (aucell_df["treatmentPhase"] == phase).sum() > 0]
    if len(groups) >= 2:
        stat, pval = kruskal(*groups)
        # Mean per phase
        means = {phase: aucell_df.loc[aucell_df["treatmentPhase"] == phase, ereg].mean()
                 for phase in phase_order if (aucell_df["treatmentPhase"] == phase).sum() > 0}
        results.append({
            "eRegulon": ereg,
            "kruskal_stat": stat,
            "pval": pval,
            **{f"mean_{phase}": means.get(phase, np.nan) for phase in phase_order},
        })

results_df = pd.DataFrame(results)
results_df["fdr"] = multipletests(results_df["pval"], method="fdr_bh")[1]
results_df = results_df.sort_values("fdr")

print(f"Testing {len(results_df)} eRegulons across {len(phase_order)} treatment phases")
print(f"Significant (FDR < 0.05): {(results_df['fdr'] < 0.05).sum()}")
print(f"Significant (FDR < 0.01): {(results_df['fdr'] < 0.01).sum()}")

# Show top differentially active eRegulons
print("\nTop 20 differentially active eRegulons:")
results_df.head(20)

In [ ]:
# --- Save differential activity results ---
results_csv = os.path.join(TABLE_DIR, f"P697.{TODAY}_{COMP}_scenicplus_eRegulon_differential_activity.csv")
results_df.to_csv(results_csv, index=False)
print(f"Saved: {results_csv}")

In [ ]:
# --- Heatmap: mean eRegulon activity by treatment phase ---
# Select top differentially active eRegulons (FDR < 0.05)
sig_eregs = results_df[results_df["fdr"] < 0.05]["eRegulon"].values

if len(sig_eregs) > 0:
    # Limit to top 50 for readability
    top_eregs = sig_eregs[:min(50, len(sig_eregs))]

    # Mean AUCell by phase
    mean_cols = [f"mean_{phase}" for phase in phase_order]
    heatmap_data = results_df[results_df["eRegulon"].isin(top_eregs)].set_index("eRegulon")[mean_cols]
    heatmap_data.columns = phase_order

    # Z-score across phases for each eRegulon
    heatmap_z = heatmap_data.subtract(heatmap_data.mean(axis=1), axis=0).div(heatmap_data.std(axis=1), axis=0)

    fig, ax = plt.subplots(1, 1, figsize=(8, max(6, len(top_eregs) * 0.25)))
    sns.heatmap(
        heatmap_z, cmap="RdBu_r", center=0, ax=ax,
        xticklabels=True, yticklabels=True,
        linewidths=0.5, linecolor="white",
    )
    ax.set_title(f"{COMP} — Top Differentially Active eRegulons by Treatment Phase\n(Z-scored mean AUCell)")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"P697.{TODAY}_{COMP}_eRegulon_activity_heatmap.pdf"), bbox_inches="tight")
    plt.show()
else:
    print("No significant eRegulons at FDR < 0.05")

In [ ]:
# --- Violin plots: top eRegulons by treatment phase ---
# Select top 12 by FDR
top_n = min(12, len(sig_eregs)) if len(sig_eregs) > 0 else 0

if top_n > 0:
    top_eregs_violin = results_df["eRegulon"].values[:top_n]
    n_cols = 4
    n_rows = int(np.ceil(top_n / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3.5))
    axes = axes.flatten()

    for i, ereg in enumerate(top_eregs_violin):
        fdr = results_df.loc[results_df["eRegulon"] == ereg, "fdr"].values[0]
        sns.violinplot(
            data=aucell_df, x="treatmentPhase", y=ereg,
            order=phase_order, ax=axes[i], inner="box", cut=0,
        )
        axes[i].set_title(f"{ereg}\n(FDR={fdr:.2e})", fontsize=9)
        axes[i].set_xlabel("")
        axes[i].set_ylabel("AUCell")
        axes[i].tick_params(axis="x", rotation=45, labelsize=7)

    # Hide empty axes
    for j in range(top_n, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(f"{COMP} — Top eRegulon Activity by Treatment Phase", fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"P697.{TODAY}_{COMP}_eRegulon_violins.pdf"), bbox_inches="tight")
    plt.show()

In [ ]:
# --- Regulon Specificity Score (RSS): phase-specific eRegulons ---
from scenicplus.RSS import regulon_specificity_scores

# RSS treats each phase as a "cell type"
rss_phase = regulon_specificity_scores(
    aucell_gene_direct,
    cell_type_key="treatmentPhase",
)

print(f"RSS matrix: {rss_phase.shape} (eRegulons × phases)")
rss_phase.head(10)

In [ ]:
# --- RSS dot plot: top phase-specific eRegulons ---
from scenicplus.plotting import plot_rss

fig, axes = plt.subplots(1, len(phase_order), figsize=(len(phase_order) * 4, 6))

for i, phase in enumerate(phase_order):
    top_rss = rss_phase[phase].sort_values(ascending=False).head(10)
    plot_rss(
        rss_phase,
        cell_type=phase,
        top_n=10,
        ax=axes[i],
    )
    axes[i].set_title(f"{phase}\nTop 10 specific eRegulons")

plt.suptitle(f"{COMP} — Regulon Specificity Scores by Treatment Phase", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"P697.{TODAY}_{COMP}_eRegulon_RSS.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# --- Per-cluster stratified analysis (optional) ---
# Test differential eRegulon activity within each WNN cluster across phases

clusters = sorted(aucell_gene_direct.obs["wnn_clusters"].unique())
print(f"WNN clusters: {clusters}")

cluster_results = []
for cl in clusters:
    mask = aucell_gene_direct.obs["wnn_clusters"] == cl
    subset = aucell_df[mask.values]
    for ereg in aucell_gene_direct.var_names:
        groups = [subset.loc[subset["treatmentPhase"] == phase, ereg].values
                  for phase in phase_order if (subset["treatmentPhase"] == phase).sum() > 0]
        if len(groups) >= 2 and all(len(g) > 0 for g in groups):
            stat, pval = kruskal(*groups)
            cluster_results.append({
                "cluster": cl, "eRegulon": ereg,
                "kruskal_stat": stat, "pval": pval,
            })

cluster_results_df = pd.DataFrame(cluster_results)
if len(cluster_results_df) > 0:
    cluster_results_df["fdr"] = multipletests(cluster_results_df["pval"], method="fdr_bh")[1]
    cluster_results_df = cluster_results_df.sort_values("fdr")

    # Save
    cluster_csv = os.path.join(
        TABLE_DIR, f"P697.{TODAY}_{COMP}_scenicplus_eRegulon_per_cluster_activity.csv"
    )
    cluster_results_df.to_csv(cluster_csv, index=False)

    # Summary
    for cl in clusters:
        n_sig = (cluster_results_df[cluster_results_df["cluster"] == cl]["fdr"] < 0.05).sum()
        print(f"  Cluster {cl}: {n_sig} significant eRegulons (FDR < 0.05)")

---
## 6. eRegulon Network Visualization

In [ ]:
# --- Heatmap-dotplot: gene AUC color × region AUC size ---
# Use top eRegulons, colored by phase-specific activity

if "direct_region_based_AUC" in scplus_mdata.mod:
    aucell_region_direct = scplus_mdata.mod["direct_region_based_AUC"].copy()
    aucell_region_direct.obs["treatmentPhase"] = cell_meta["treatmentPhase"].values

    # Select top 20 eRegulons from our differential test
    top_20 = results_df["eRegulon"].values[:20]
    # Find matching eRegulons in both gene and region AUCell
    common_eregs = list(set(top_20) & set(aucell_gene_direct.var_names) & set(aucell_region_direct.var_names))
    common_eregs = [e for e in top_20 if e in common_eregs]  # preserve order

    if len(common_eregs) > 0:
        from scenicplus.plotting import heatmap_dotplot

        heatmap_dotplot(
            scplus_mdata,
            selected_regulons=common_eregs[:15],
            group_variable="treatmentPhase",
        )
        plt.savefig(os.path.join(FIG_DIR, f"P697.{TODAY}_{COMP}_eRegulon_heatmap_dotplot.pdf"), bbox_inches="tight")
        plt.show()
    else:
        print("No common eRegulons between gene and region AUCell for dotplot")
else:
    print("Region-based AUCell not available in MuData")

In [ ]:
# --- UMAP colored by top eRegulon activity ---
# Show top 4 phase-specific eRegulons on the original RNA UMAP
top4 = results_df["eRegulon"].values[:4]

# Transfer AUCell scores to the RNA AnnData UMAP
for ereg in top4:
    if ereg in aucell_gene_direct.var_names:
        adata_rna.obs[ereg] = aucell_gene_direct[:, ereg].X.toarray().flatten() \
            if scipy.sparse.issparse(aucell_gene_direct.X) \
            else aucell_gene_direct[:, ereg].X.flatten()

fig, axes = plt.subplots(1, len(top4), figsize=(len(top4) * 5, 4))
for i, ereg in enumerate(top4):
    if ereg in adata_rna.obs.columns:
        sc.pl.umap(adata_rna, color=ereg, ax=axes[i], show=False,
                   title=ereg, color_map="viridis", size=10)

plt.suptitle(f"{COMP} — Top eRegulon Activity on RNA UMAP", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"P697.{TODAY}_{COMP}_eRegulon_activity_umap.pdf"), bbox_inches="tight")
plt.show()

---
## 7. Save Results

In [ ]:
# --- Export AUCell scores per cell with metadata ---
export_df = aucell_df.copy()
export_df["donorID"] = aucell_gene_direct.obs["donorID"].values
export_df["wnn_clusters"] = aucell_gene_direct.obs["wnn_clusters"].values

aucell_csv = os.path.join(TABLE_DIR, f"P697.{TODAY}_{COMP}_scenicplus_AUCell_scores.csv")
export_df.to_csv(aucell_csv)
print(f"Saved AUCell scores: {aucell_csv}")

# --- Export RSS table ---
rss_csv = os.path.join(TABLE_DIR, f"P697.{TODAY}_{COMP}_scenicplus_RSS.csv")
rss_phase.to_csv(rss_csv)
print(f"Saved RSS table: {rss_csv}")

# --- Export eRegulon tables (copy from pipeline output) ---
for key, fname in [("eRegulons_direct", "eRegulon_direct"), ("eRegulons_extended", "eRegulons_extended")]:
    src = os.path.join(scplus_pipeline_dir, config["output_data"][key])
    dst = os.path.join(TABLE_DIR, f"P697.{TODAY}_{COMP}_scenicplus_{fname}.tsv")
    if os.path.exists(src):
        import shutil
        shutil.copy2(src, dst)
        print(f"Copied: {dst}")

print("\nDone! All results exported.")

---
## 8. Session Info

In [ ]:
session_info.show()